In [5]:
import numpy as np
from scipy.optimize import linprog
from numpy.linalg import eigh
from scipy.sparse import diags

In [36]:
def generate_parameters(n, cut_through_box=True):
    c = np.random.uniform(-1, 1, n)  # c can be any value in R^n, including negative
    a = np.random.rand(n) + 1e-6  # ensure a is strictly positive
    l = np.random.rand(n)  # l can be any non-negative value in R^n
    u = l + np.random.rand(n)  # ensure u is greater than l
    
    if cut_through_box:
        # Choose 'b' such that the hyperplane a^T x = b cuts through the box [l, u].
        b = np.dot(a, (l + u) / 2)  # This ensures the hyperplane passes through the center of the box
    else:
        b = np.random.rand(1)[0] + 1e-6  # ensure b is strictly positive, scalar

    return c, a, b, l, u


(array([-0.06685031,  0.44346203, -0.33445575,  0.47795302,  0.37584529]),
 array([0.15068375, 0.47685604, 0.60930039, 0.82912079, 0.93534649]),
 2.617254716031496,
 array([0.02659444, 0.95769074, 0.36071373, 0.56941685, 0.72662255]),
 array([0.93326948, 1.19420568, 0.83881568, 1.17057172, 1.29422675]))

In [41]:
def linear_subproblem(c, a, b, l, u):
    n = len(c)
    x_star = []

    for i in range(n):
        if c[i] >= 0:
            x_star.append(l[i])
        else:
            x_star.append(u[i])
    
    if (a @ x_star) >= b:
        return x_star
    
    # The minimum value of the objective function is not in feasible region
    print("Not in the vertex of the hyperbox")
    x_pack = [(i, x_star[i], c[i], a[i]) for i in range(n)]
    x_pack = sorted(x_pack, key=lambda t: t[2]/t[3])
    ordered_index = [t[0] for t in x_pack]
    
    for idx in ordered_index:
        x_star[idx] = u[idx]

        if (a @ x_star) > b:
            x_star[idx] = (b - (a @ x_star) + (a[idx] * x_star[idx])) / a[idx]
            return x_star
    
    print("No solution found")
    return x_star
    

In [45]:
tries = 1000
failed = 0

for i in range(tries):
    c, a, b, l, u = generate_parameters(100)

    A = -np.array([a])
    b_ = -np.array(b)
    bounds = [] 
    for i in range(len(l)):
        bounds.append((l[i], u[i]))

    solution = linear_subproblem(c, a, b, l, u)
    golden = linprog(c, A_ub=A, b_ub=b_, bounds=bounds, method='highs')
    golden = golden.x

    if not np.allclose(solution, golden):
        print("Failed for seed", i)
        failed += 1

print(f"Failed {failed} out of {tries} tries")

Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the vertex of the hyperbox
Not in the ver